# Encoder-Only — Hyperparameter Sweep (Lorenz)

Performs a **Hydra grid sweep** over encoder-only regularisation parameters
using SLURM for parallel training.

**Pipeline:**
1. Set hyperparameters in clearly labelled cells (Sections 1–2)
2. Build Hydra config (single source of truth for local + sweep)
3. Generate data locally for post-hoc diagnostics
4. Check W&B for already-completed runs
5. Launch Hydra `--multirun` sweep via SLURM (one job per parameter combination)
6. Collect results and visualise

> **Wait for all SLURM jobs to finish** before running Section 7.

In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
import itertools
import subprocess
import sys, os

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import wandb
from hydra.utils import instantiate
from omegaconf import OmegaConf

from JacobianODE.jacobians.core import seed_everything
from JacobianODE.jacobians.data import make_trajectories, postprocess_data, create_dataloaders
from JacobianODE.jacobians.training import train_model
from JacobianODE.encoder_only.config import load_encoder_config

torch.set_float32_matmul_precision('high')
print(f'PyTorch {torch.__version__}  |  GPUs: {torch.cuda.device_count()}')

## 1. Hyperparameters

In [ ]:
# ----------------------------------------------------------------
# Paths and W&B settings
# ----------------------------------------------------------------
SAVE_DIR = "/orcd/data/ekmiller/001/eisenaj/JacobianODE/lightning/encoder_runs"
WANDB_ENTITY = "JacobianODE"
WANDB_PROJECT = None  # set to None for auto-generated name
WANDB_PROJECT_PATH = f"{WANDB_ENTITY}/{WANDB_PROJECT}" if WANDB_PROJECT else None

In [ ]:
# ----------------------------------------------------------------
# Data hyperparameters
# ----------------------------------------------------------------
NUM_ICS = 32
N_PERIODS = 12
PTS_PER_PERIOD = 100
# SEQ_LENGTH = 50
SEQ_LENGTH = 100
# OBS_NOISE = 0.05
OBS_NOISE = 0.01

# Partial observation (delay embedding)
# OBSERVED_INDICES = [0, 1, 2]   # Observe ALL three Lorenz dimensions; use [0] for x-only
OBSERVED_INDICES = [0]
# N_DELAYS         = 1           # Number of delays for delay embedding (1 = no embedding)
# N_DELAYS         = 43
DELAY_SPACING    = 1           # Spacing between delays when N_DELAYS > 1
# N_DELAYS = 100
N_DELAYS = 25
# DELAY_SPACING = 10

In [ ]:
# ----------------------------------------------------------------
# Encoder / architecture hyperparameters
# ----------------------------------------------------------------
# MODEL = 'ssm'              # SSM (LRU) sequence encoder
# MODEL = 'transformer'      # Transformer sequence encoder
# MODEL = 'tcn'              # TCN sequence encoder
# MODEL = 'coupling'         # AffineCouplingEncoder (invertible, dim-preserving)
MODEL = 'spline_coupling'    # CouplingEncoder with rational quadratic splines + ActNorm

assert MODEL in ('ssm', 'transformer', 'tcn', 'coupling', 'spline_coupling'), f"Invalid MODEL: {MODEL}"
IS_COUPLING = MODEL in ('coupling', 'spline_coupling')

# ============================================================
# Sequence encoder settings (ssm / transformer / tcn)
# ============================================================
if not IS_COUPLING:
    N_LATENT = 10

    # SSM (LRU) parameters -- used when MODEL='ssm'
    SSM_D_MODEL    = 128
    SSM_D_STATE    = 64
    SSM_N_LAYERS   = 3
    SSM_R_MIN      = 0.9
    SSM_R_MAX      = 0.999
    SSM_FFN_EXPAND = 4
    SSM_DROPOUT    = 0.1

    # Transformer parameters -- used when MODEL='transformer'
    TRANSFORMER_D_MODEL       = 64
    TRANSFORMER_N_HEADS       = 4
    TRANSFORMER_N_LAYERS      = 3
    TRANSFORMER_DIM_FEEDFORWARD = 128
    TRANSFORMER_DROPOUT       = 0.1

    # TCN parameters -- used when MODEL='tcn'
    TCN_N_CHANNELS = 64
    TCN_KERNEL_SIZE = 7
    TCN_N_LAYERS   = 4
    TCN_DROPOUT    = 0.1

    # Decoder head (shared across sequence architectures)
    USE_SAME_STATE_DECODER = True
    USE_NEXT_STATE_DECODER = True
    DECODER_HIDDEN_DIM    = 128
    DECODER_N_LAYERS      = 2
    NEXT_STATE_BURN_IN    = 0

    # Multi-step ahead prediction
    K_STEPS_AHEAD = 10
    N_OBS_PRED = len(OBSERVED_INDICES)

# ============================================================
# Coupling flow encoder settings (coupling / spline_coupling)
# Self-supervised: encode → split → zero-pad → inverse decode → MSE
# ============================================================
if IS_COUPLING:
    # ---- Architecture ----
    N_COUPLING_LAYERS   = 8      # number of coupling stacks
    COUPLING_HIDDEN_DIM = 128    # conditioner MLP width
    N_HIDDEN_LAYERS     = 2      # conditioner MLP depth
    ZERO_INIT           = True   # identity initialization trick
    PERMUTATION_SEED    = 42     # base seed for fixed random permutations
    USE_LOFT            = False  # LOFT layer after coupling blocks
    LOFT_TAU            = 100.0  # LOFT threshold

    # Affine coupling only (ignored for spline_coupling)
    SCALE_ACTIVATION    = "tanh"       # bounds log-scale via tanh
    SCALE_CLAMP         = 3.0          # max |log_s|
    CLAMP_TYPE          = "symmetric"  # 'symmetric' | 'asymmetric'
    ALPHA_POS           = 0.1          # asymmetric clamp: expansion bound
    ALPHA_NEG           = 2.0          # asymmetric clamp: compression bound

    # Spline coupling only (ignored for affine coupling)
    NUM_BINS            = 8      # rational quadratic spline segments
    TAIL_BOUND          = 3.0    # linear tails outside [-B, B]
    USE_ACTNORM         = True   # ActNorm between coupling layers

    # ---- Subspace splitting ----
    N_TARGET_DIMS        = 3           # dynamic subspace (e.g. 3 for Lorenz)
    KL_DIVERGENCE_WEIGHT = 1.0         # unified KL weight (null MSE + optional VAE KL)
    RECONSTRUCTION_MODE  = "most_recent"  # 'uniform' | 'harmonic' | 'most_recent'

    # ---- VAE settings ----
    USE_VAE                = False     # VAE reparameterization on dynamic subspace
    KL_WARMUP_EPOCHS       = 0        # 0 = fixed weight, >0 = linear ramp

print(f"Encoder: {MODEL}" + (f" (n_target_dims={N_TARGET_DIMS})" if IS_COUPLING else f" (n_latent={N_LATENT})"))

In [27]:
# ----------------------------------------------------------------
# Training hyperparameters
# ----------------------------------------------------------------
BATCH_SIZE = 32
LEARNING_RATE = 1e-4
MAX_EPOCHS = 150
LIMIT_TRAIN_BATCHES = 200
EARLY_STOPPING_PATIENCE = 2
PERCENT_THRESH = 0.01

In [ ]:
# ----------------------------------------------------------------
# Sweep parameters -- Hydra --multirun will grid over all combinations
# ----------------------------------------------------------------
if IS_COUPLING:
    # Coupling flow sweep: KL divergence weight is the primary regularizer
    SWEEP_PARAMS = {
        "model.kl_divergence_weight": [0, 0.01, 0.1, 1.0, 10.0],
        # "model.reconstruction_mode": ["uniform", "most_recent"],
        # "model.decoder_recon_weight": [0.0, 0.1, 1.0],
    }
else:
    # Sequence encoder sweep: FNN weight is the primary regularizer
    SWEEP_PARAMS = {
        "training.lightning.fnn_weight": [0, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1, 10],
        # "training.lightning.fnn_normalize": [False, True],
        "training.lightning.fnn_normalize": [True],
        # "training.lightning.fnn_elementwise_regularization": [False, True],
        "training.lightning.fnn_elementwise_regularization": [True],
        # "training.lightning.amplification_weight": [0.0, 0.001, 0.01],
        # "training.lightning.decov_weight": [0.0, 0.001],
        "training.lightning.amplification_weight": [0.0],
        "training.lightning.decov_weight": [0.0],
    }

# Count total combinations
n_combos = 1
for vals in SWEEP_PARAMS.values():
    n_combos *= len(vals)
print(f"Sweep grid: {n_combos} total combinations")
for k, v in SWEEP_PARAMS.items():
    print(f"  {k}: {v}")

## 2. Build Base Config

Build the Hydra overrides list — this is the **single source of truth** used both
for local data generation and as the template for the sweep command.

In [ ]:
# Format list-valued overrides without spaces so they double as valid Hydra CLI args
_obs_idx_str = str(list(OBSERVED_INDICES)).replace(' ', '')

overrides = [
    # --- Model ---
    f"model={MODEL}",
]

# ============================================================
# Coupling flow encoder overrides
# ============================================================
if IS_COUPLING:
    _n_raw_obs = len(OBSERVED_INDICES)
    _n_input = N_DELAYS * _n_raw_obs

    overrides.extend([
        f"++model.encoder.n_input={_n_input}",
        f"++model.encoder.n_coupling_layers={N_COUPLING_LAYERS}",
        f"++model.encoder.hidden_dim={COUPLING_HIDDEN_DIM}",
        f"++model.encoder.n_hidden_layers={N_HIDDEN_LAYERS}",
        f"++model.encoder.zero_init={str(ZERO_INIT).lower()}",
        f"++model.encoder.permutation_seed={PERMUTATION_SEED}",
        f"++model.encoder.use_loft={str(USE_LOFT).lower()}",
        f"++model.encoder.loft_tau={LOFT_TAU}",
    ])

    if MODEL == 'coupling':
        overrides.extend([
            f"++model.encoder.scale_activation={SCALE_ACTIVATION}",
            f"++model.encoder.scale_clamp={SCALE_CLAMP}",
            f"++model.encoder.clamp_type={CLAMP_TYPE}",
            f"++model.encoder.alpha_pos={ALPHA_POS}",
            f"++model.encoder.alpha_neg={ALPHA_NEG}",
        ])
    elif MODEL == 'spline_coupling':
        overrides.extend([
            f"++model.encoder.num_bins={NUM_BINS}",
            f"++model.encoder.tail_bound={TAIL_BOUND}",
            f"++model.encoder.use_actnorm={str(USE_ACTNORM).lower()}",
        ])

    overrides.extend([
        f"++model.n_target_dims={N_TARGET_DIMS}",
        f"++model.kl_divergence_weight={KL_DIVERGENCE_WEIGHT}",
        f"++model.reconstruction_mode={RECONSTRUCTION_MODE}",
        f"++model.use_vae={str(USE_VAE).lower()}",
        f"++model.kl_warmup_epochs={KL_WARMUP_EPOCHS}",
    ])

# ============================================================
# Sequence encoder overrides (ssm / transformer / tcn)
# ============================================================
else:
    overrides.extend([
        f"++model.encoder.n_latent={N_LATENT}",
        f"++model.use_same_state_decoder={str(USE_SAME_STATE_DECODER).lower()}",
        f"++model.use_next_state_decoder={str(USE_NEXT_STATE_DECODER).lower()}",
        f"++model.decoder_hidden_dim={DECODER_HIDDEN_DIM}",
        f"++model.decoder_n_layers={DECODER_N_LAYERS}",
        f"++model.next_state_burn_in={NEXT_STATE_BURN_IN}",
        f"++model.k_steps_ahead={K_STEPS_AHEAD}",
        f"++model.n_obs_pred={N_OBS_PRED}",
    ])

    if MODEL == 'ssm':
        overrides.extend([
            f"++model.encoder.d_model={SSM_D_MODEL}",
            f"++model.encoder.d_state={SSM_D_STATE}",
            f"++model.encoder.n_layers={SSM_N_LAYERS}",
            f"++model.encoder.r_min={SSM_R_MIN}",
            f"++model.encoder.r_max={SSM_R_MAX}",
            f"++model.encoder.ffn_expand={SSM_FFN_EXPAND}",
            f"++model.encoder.dropout={SSM_DROPOUT}",
        ])
    elif MODEL == 'transformer':
        overrides.extend([
            f"++model.encoder.d_model={TRANSFORMER_D_MODEL}",
            f"++model.encoder.n_heads={TRANSFORMER_N_HEADS}",
            f"++model.encoder.n_layers={TRANSFORMER_N_LAYERS}",
            f"++model.encoder.dim_feedforward={TRANSFORMER_DIM_FEEDFORWARD}",
            f"++model.encoder.dropout={TRANSFORMER_DROPOUT}",
        ])
    elif MODEL == 'tcn':
        overrides.extend([
            f"++model.encoder.n_channels={TCN_N_CHANNELS}",
            f"++model.encoder.kernel_size={TCN_KERNEL_SIZE}",
            f"++model.encoder.n_layers={TCN_N_LAYERS}",
            f"++model.encoder.dropout={TCN_DROPOUT}",
        ])

# ============================================================
# Shared overrides (training, data, paths)
# ============================================================
overrides.extend([
    # --- Training ---
    f"++training.batch_size={BATCH_SIZE}",
    f"++training.lightning.optimizer_kwargs.lr={LEARNING_RATE}",
    f"++training.trainer_params.max_epochs={MAX_EPOCHS}",
    f"++training.trainer_params.limit_train_batches={LIMIT_TRAIN_BATCHES}",
    f"++training.early_stopping.early_stopping_patience={EARLY_STOPPING_PATIENCE}",
    f"++training.early_stopping.percent_thresh={PERCENT_THRESH}",

    # --- Data ---
    f"++data.trajectory_params.num_ics={NUM_ICS}",
    f"++data.trajectory_params.n_periods={N_PERIODS}",
    f"++data.trajectory_params.pts_per_period={PTS_PER_PERIOD}",
    f"++data.train_test_params.seq_length={SEQ_LENGTH}",
    f"++data.postprocessing.obs_noise={OBS_NOISE}",

    # --- Partial observation / delay embedding ---
    f"++data.train_test_params.delay_embedding_params.observed_indices={_obs_idx_str}",
    f"++data.train_test_params.delay_embedding_params.n_delays={N_DELAYS}",
    f"++data.train_test_params.delay_embedding_params.delay_spacing={DELAY_SPACING}",

    # --- Save directory ---
    f"++training.logger.save_dir={SAVE_DIR}",
])

cfg = load_encoder_config(overrides=overrides)

# Auto-generate project name if not set
if WANDB_PROJECT is None:
    data_cls = cfg.data.flow._target_.split('.')[-1]
    if IS_COUPLING:
        WANDB_PROJECT = f"{data_cls}__{MODEL}__EncoderOnly"
    else:
        WANDB_PROJECT = f"{data_cls}__EncoderOnly"
    WANDB_PROJECT_PATH = f"{WANDB_ENTITY}/{WANDB_PROJECT}"

print(f"W&B project: {WANDB_PROJECT_PATH}")
enc_target = cfg.model.encoder._target_.split('.')[-1]
print(f"Encoder: {enc_target}")
if IS_COUPLING:
    print(f"  n_target_dims={N_TARGET_DIMS}, kl_weight={KL_DIVERGENCE_WEIGHT}, recon_mode={RECONSTRUCTION_MODE}")
    if USE_VAE:
        print(f"  VAE: enabled (kl_warmup={KL_WARMUP_EPOCHS})")
else:
    print(f"  n_latent={cfg.model.encoder.n_latent}, d_model={cfg.model.encoder.d_model}")
print(OmegaConf.to_yaml(cfg))

## 3. Generate Data

Generate trajectories once locally. The same data pipeline runs inside each
SLURM job (deterministic via fixed seed), but we also need it here for
post-hoc diagnostics.

In [30]:
seed_everything(cfg.data.flow.random_state)
eq, sol, dt = make_trajectories(cfg)
print(f"Full trajectory shape: {sol['values'].shape}")
print(f"Time step dt = {dt:.4f}")

Full trajectory shape: (32, 1200, 3)
Time step dt = 0.0150


In [31]:
result = postprocess_data(cfg, sol['values'])
values = result.values
cfg.data.postprocessing.noise_scale_factor = result.noise_scale_factor
cfg.data.postprocessing.mu = result.mu
cfg.data.postprocessing.sigma = result.sigma

train_dl, val_dl, test_dl, trajs = create_dataloaders(cfg, values)
n_obs = trajs['train_trajs'].sequence.shape[-1]

print(f"n_obs = {n_obs}")
print(f"Train: {len(train_dl.dataset)} sequences")
print(f"Val:   {len(val_dl.dataset)} sequences")

n_obs = 5
Train: 23320 sequences
Val:   7420 sequences


## 4. Check W&B for Completed Runs

Query W&B for runs that match the current sweep parameter combinations.
Only the remaining (not-yet-completed) combinations will be launched.

In [32]:
def _get_sweep_param_from_run(run, key):
    """Extract a sweep parameter value from a W&B run config.

    Keys are dot-separated Hydra paths like 'training.lightning.fnn_weight'.
    """
    parts = key.split('.')
    val = run.config
    for p in parts:
        if not isinstance(val, dict) or p not in val:
            return None
        val = val[p]
    return val


def _combo_matches_run(combo, run):
    """Check if a parameter combination matches a finished W&B run."""
    for key, target_val in combo.items():
        run_val = _get_sweep_param_from_run(run, key)
        if run_val is None:
            return False
        # Handle bool vs str comparisons
        if isinstance(target_val, bool):
            if bool(run_val) != target_val:
                return False
        elif isinstance(target_val, (int, float)):
            if abs(float(run_val) - float(target_val)) > 1e-10:
                return False
        else:
            if str(run_val) != str(target_val):
                return False
    return True


# Build all parameter combinations
sweep_keys = list(SWEEP_PARAMS.keys())
sweep_value_lists = [SWEEP_PARAMS[k] for k in sweep_keys]
all_combos = [
    dict(zip(sweep_keys, vals))
    for vals in itertools.product(*sweep_value_lists)
]

# Query W&B
api = wandb.Api()
try:
    existing_runs = api.runs(WANDB_PROJECT_PATH)
    print(f"Found {len(existing_runs)} total runs in {WANDB_PROJECT_PATH}")
except Exception as e:
    print(f"Could not query project (may not exist yet): {e}")
    existing_runs = []

finished_runs = [r for r in existing_runs if r.state == 'finished']

already_done = []
remaining_combos = []
for combo in all_combos:
    if any(_combo_matches_run(combo, r) for r in finished_runs):
        already_done.append(combo)
    else:
        remaining_combos.append(combo)

print(f"\nAlready completed: {len(already_done)} / {len(all_combos)}")
print(f"Remaining to run:  {len(remaining_combos)}")

Could not query project (may not exist yet): Could not find project Lorenz_N5T10__EncoderOnly

Already completed: 0 / 8
Remaining to run:  8


## 5. Launch Hydra Grid Sweep

Uses Hydra `--multirun` with the SLURM launcher to submit one job per
parameter combination. The `overrides` list from Section 2 is the
**single source of truth** — the sweep command simply appends the swept
parameters (comma-separated) and W&B/SLURM settings.

> **Wait for all SLURM jobs to finish** before proceeding to Section 7.

In [ ]:
# ----------------------------------------------------------------
# Build the Hydra --multirun sweep command
# ----------------------------------------------------------------
if remaining_combos:
    # Hydra --multirun creates the full grid from comma-separated values.
    # We pass the complete sweep values; the run_encoder entry point will
    # train each combination as a separate SLURM job.

    # Remove swept keys from fixed overrides to avoid conflicts
    _swept_keys = set(SWEEP_PARAMS.keys())
    def _is_swept(ov):
        for k in _swept_keys:
            # Match both "key=" and "++key=" forms
            bare = ov.lstrip('+')
            if bare.startswith(k + "="):
                return True
        return False
    fixed_overrides = [ov for ov in overrides if not _is_swept(ov)]

    n_remaining = len(remaining_combos)
    use_subset_sweep = n_remaining < len(all_combos)

    if use_subset_sweep:
        # One command per remaining combo
        sweep_cmds = []
        for combo in remaining_combos:
            combo_overrides = [
                f"++{key}={str(val).lower() if isinstance(val, bool) else val}"
                for key, val in combo.items()
            ]
            cmd_overrides = fixed_overrides + combo_overrides + [
                f"wandb_entity={WANDB_ENTITY}",
                f"wandb_project={WANDB_PROJECT}",
                "slurm=default",
            ]
            sweep_cmds.append(
                "python -m JacobianODE.encoder_only.run_encoder --multirun "
                + " ".join(cmd_overrides)
            )
    else:
        # Full grid sweep: comma-separated values
        sweep_param_overrides = []
        for key, vals in SWEEP_PARAMS.items():
            val_str = ",".join(
                str(v).lower() if isinstance(v, bool) else str(v)
                for v in vals
            )
            sweep_param_overrides.append(f"++{key}={val_str}")

        cmd_overrides = fixed_overrides + sweep_param_overrides + [
            f"wandb_entity={WANDB_ENTITY}",
            f"wandb_project={WANDB_PROJECT}",
            "slurm=default",
        ]
        sweep_cmds = [
            "python -m JacobianODE.encoder_only.run_encoder --multirun "
            + " ".join(cmd_overrides)
        ]

    sweep_type = "subset (1 cmd per combo)" if use_subset_sweep else "grid"
    print(f"Sweep: {n_remaining} jobs ({sweep_type})")
    print(f"\nFirst command:\n{sweep_cmds[0][:300]}...")
else:
    sweep_cmds = []
    print("No sweep to launch -- all runs already completed.")

In [ ]:
# Launch the sweep (submits SLURM jobs).
if sweep_cmds:
    if len(sweep_cmds) > 1:
        print(f"Starting {len(sweep_cmds)} commands in parallel...")
    procs = [subprocess.Popen(cmd, shell=True) for cmd in sweep_cmds]
    for i, p in enumerate(procs):
        p.wait()
        if p.returncode != 0 and len(procs) > 1:
            print(f"Command {i + 1}/{len(procs)} exited with code {p.returncode}")
    print("Sweep complete.")
else:
    print("No sweep to launch -- all runs already completed.")

## 6. Collect W&B Runs

After all jobs finish, re-query W&B to verify all runs completed successfully.
Crashed/failed runs are flagged.

In [35]:
api = wandb.Api()
all_runs = api.runs(WANDB_PROJECT_PATH)
print(f"Found {len(all_runs)} total runs in {WANDB_PROJECT_PATH}")

# Flag crashed/failed runs
for run in all_runs:
    if run.state in ('crashed', 'failed'):
        print(f"  CRASHED/FAILED: {run.id} ({run.name}) — state={run.state}")

# Check completeness
finished_runs = [r for r in all_runs if r.state == 'finished']
completed_combos = []
missing_combos = []
for combo in all_combos:
    if any(_combo_matches_run(combo, r) for r in finished_runs):
        completed_combos.append(combo)
    else:
        missing_combos.append(combo)

print(f"\nCompleted: {len(completed_combos)} / {len(all_combos)}")
if missing_combos:
    print(f"MISSING {len(missing_combos)} combinations — re-run Section 5 to retry.")
    for combo in missing_combos[:5]:
        print(f"  {combo}")
    if len(missing_combos) > 5:
        print(f"  ... and {len(missing_combos) - 5} more")
else:
    print("All combinations completed successfully!")

Found 8 total runs in JacobianODE/Lorenz_N5T10__EncoderOnly

Completed: 8 / 8
All combinations completed successfully!


## 7. Analyse Sweep Results

In [ ]:
# ----------------------------------------------------------------
# Build results DataFrame from finished W&B runs
# ----------------------------------------------------------------
records = []
for r in finished_runs:
    # Check this run matches one of our sweep combos
    if not any(_combo_matches_run(combo, r) for combo in all_combos):
        continue

    rec = {}
    # Extract sweep parameters
    for key in sweep_keys:
        short_name = key.split('.')[-1]  # e.g. 'fnn_weight' or 'kl_divergence_weight'
        rec[short_name] = _get_sweep_param_from_run(r, key)

    # Extract metrics (coupling vs sequence have different metric keys)
    if IS_COUPLING:
        rec['val_recon_loss']   = r.summary.get('val/recon_loss', float('nan'))
        rec['val_kl_null_loss'] = r.summary.get('val/kl_null_loss', float('nan'))
        rec['val_kl_dyn_loss']  = r.summary.get('val/kl_dyn_loss', float('nan'))
        rec['val_kl_total']     = r.summary.get('val/kl_total_loss', float('nan'))
        rec['val_total_loss']   = r.summary.get('val/total_loss', r.summary.get('mean val loss', float('nan')))
    else:
        rec['val_same_loss']    = r.summary.get('val/same_state_loss', float('nan'))
        rec['val_next_loss']    = r.summary.get('val/next_state_loss', float('nan'))
        rec['val_fnn_loss']     = r.summary.get('val/fnn_loss', float('nan'))
        rec['val_amp_loss']     = r.summary.get('val/amplification_loss', float('nan'))
        rec['latent_util']      = r.summary.get('val/latent_utilization', float('nan'))
        rec['val_total_loss']   = r.summary.get('val/total_loss', float('nan'))

    rec['run_name'] = r.name
    rec['run_id']   = r.id
    records.append(rec)

sort_col = 'val_recon_loss' if IS_COUPLING else 'val_same_loss'
df = pd.DataFrame(records).sort_values(sort_col)
print(f"{len(df)} finished runs in results")
df.head(10)

In [ ]:
# ---- Scatter: sweep param vs primary loss ----
fig, ax = plt.subplots(figsize=(7, 5))

if IS_COUPLING:
    x_col = 'kl_divergence_weight'
    y_col = 'val_recon_loss'
    sc = ax.scatter(df[x_col], df[y_col], s=60, alpha=0.8, edgecolors='k', linewidths=0.4)
    ax.set_xlabel('kl_divergence_weight')
    ax.set_ylabel('val/recon_loss')
    ax.set_title('Sweep: KL weight vs. reconstruction loss')
else:
    x_col = 'fnn_weight'
    y_col = 'val_same_loss'
    sc = ax.scatter(
        df[x_col], df[y_col],
        c=df['amplification_weight'], cmap='viridis',
        s=60, alpha=0.8, edgecolors='k', linewidths=0.4,
    )
    plt.colorbar(sc, ax=ax, label='amplification_weight')
    ax.set_xlabel('fnn_weight')
    ax.set_ylabel('val/same_state_loss')
    ax.set_title('Sweep: regularisation vs. reconstruction loss')

ax.set_xscale('symlog', linthresh=1e-4)
plt.tight_layout()
plt.show()

In [ ]:
# ---- Scatter: sweep param vs secondary loss ----
fig, ax = plt.subplots(figsize=(7, 5))

if IS_COUPLING:
    x_col = 'kl_divergence_weight'
    y_col = 'val_kl_null_loss'
    sc = ax.scatter(df[x_col], df[y_col], s=60, alpha=0.8, edgecolors='k', linewidths=0.4)
    ax.set_xlabel('kl_divergence_weight')
    ax.set_ylabel('val/kl_null_loss')
    ax.set_title('Sweep: KL weight vs. null-space loss')
else:
    x_col = 'fnn_weight'
    y_col = 'val_next_loss'
    sc = ax.scatter(
        df[x_col], df[y_col],
        c=df['amplification_weight'], cmap='viridis',
        s=60, alpha=0.8, edgecolors='k', linewidths=0.4,
    )
    plt.colorbar(sc, ax=ax, label='amplification_weight')
    ax.set_xlabel('fnn_weight')
    ax.set_ylabel('val/next_state_loss')
    ax.set_title('Sweep: regularisation vs. next-state loss')

ax.set_xscale('symlog', linthresh=1e-4)
plt.tight_layout()
plt.show()

In [ ]:
# ---- Loss distribution across sweep runs ----
fig, ax = plt.subplots(figsize=(6, 4))

if IS_COUPLING:
    col = 'val_recon_loss'
    ax.hist(df[col].dropna(), bins=20, color='steelblue', edgecolor='white')
    ax.set_xlabel('val/recon_loss')
    ax.set_title('Reconstruction loss across sweep runs')
else:
    col = 'latent_util'
    ax.hist(df[col].dropna(), bins=20, color='steelblue', edgecolor='white')
    ax.set_xlabel('latent utilisation (entropy-based, 0\u20131)')
    ax.set_title('Latent utilisation across sweep runs')

ax.set_ylabel('count')
plt.tight_layout()
plt.show()

In [ ]:
# ---- Best run summary ----
sort_col = 'val_recon_loss' if IS_COUPLING else 'val_same_loss'
best = df.sort_values(sort_col).iloc[0]
print(f'Best run (lowest {sort_col}):')
for col in df.columns:
    print(f'  {col:40s}: {best[col]}')